In [1]:
# %% 히트맵 생성: V9 모델로 전체 데이터 추론 → 채널별 .pt 저장
import os, json, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
HEATMAP_DIR = "./data/8_heatmap"
GRID_W = 80
GRID_H = 80
NUM_WORKERS = 32
FPS = 10
MAX_FRAMES = 2000
MAX_ACTIVE_PER_FRAME = 150
MAX_TEXT_LEN = 200
SLOT_DIM = 5
TIME_DIM = 3
PATCH_SIZE = 8
N_PATCHES_X = GRID_W // PATCH_SIZE
N_PATCHES_Y = GRID_H // PATCH_SIZE
N_PATCHES = N_PATCHES_X * N_PATCHES_Y
D_MODEL = 256
N_HEADS = 8
N_LAYERS_TEMPORAL = 4
N_LAYERS_SPATIAL = 4
D_FF = 512
DROPOUT = 0.0
INTRA_CHUNK = 512
SPATIAL_CHUNK = 128

CKPT_PATH = "./model/8_layout_vit_patch_mask_focal/epoch_012.pt"

os.makedirs(HEATMAP_DIR, exist_ok=True)

text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개")


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None
    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])
        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f
    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))
    video_name = data.get("video", "")
    file_id = os.path.basename(path)[:-5]
    inst_list = []
    for inst in instances:
        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
        })
    stt_path = os.path.join(STT_DIR, channel, file_id + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass
    return {
        "channel": channel,
        "video_name": video_name,
        "file_id": file_id,
        "instances": inst_list,
        "stt_segments": stt_segments,
        "duration": duration,
    }


json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

samples = []
channel_set = set()
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        channel_set.add(result["channel"])
        samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

# 채널별 그룹화
by_channel = {}
for s in samples:
    if s["channel"] not in by_channel:
        by_channel[s["channel"]] = []
    by_channel[s["channel"]].append(s)

print(f"✅ 전체 영상: {len(samples):,}개  채널: {len(channels)}개")


class IntraFrameModule(nn.Module):
    def __init__(self, d_model=D_MODEL):
        super().__init__()
        self.slot_proj = nn.Sequential(
            nn.Linear(SLOT_DIM, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        self.pool_query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.combine = nn.Sequential(
            nn.Linear(d_model * 3 + 1, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        self.norm = nn.LayerNorm(d_model)

    def forward(self, slot_feats, slot_mask):
        N, K, _ = slot_feats.shape
        device = slot_feats.device
        dtype = slot_feats.dtype
        tokens = self.slot_proj(slot_feats)
        d = tokens.shape[-1]
        has_any = slot_mask.any(dim=1)
        pooled = torch.zeros(N, d, device=device, dtype=dtype)
        if has_any.any():
            vi = has_any.nonzero(as_tuple=True)[0]
            vt = tokens[vi]
            vm = slot_mask[vi]
            vp = ~vm
            V = vt.shape[0]
            count = vm.sum(dim=1, keepdim=True).float()
            count_norm = count / MAX_ACTIVE_PER_FRAME
            mt = vt.masked_fill(vp.unsqueeze(-1), 0.0)
            mean_pool = mt.sum(dim=1) / count.clamp(min=1)
            mfm = vt.masked_fill(vp.unsqueeze(-1), float("-inf"))
            max_pool = mfm.max(dim=1).values
            q = self.pool_query.expand(V, -1, -1)
            a = torch.bmm(q, vt.transpose(1, 2)) / (d ** 0.5)
            a = a.masked_fill(vp.unsqueeze(1), float("-inf"))
            a = F.softmax(a, dim=-1)
            attn_pool = torch.bmm(a, vt).squeeze(1)
            combined = torch.cat([attn_pool, mean_pool, max_pool, count_norm], dim=-1)
            pooled[vi] = self.norm(self.combine(combined)).to(dtype)
        return pooled


class ViTPatchMaskModel(nn.Module):
    def __init__(self, n_channels, d_model=D_MODEL, n_heads=N_HEADS,
                 n_layers_temporal=N_LAYERS_TEMPORAL, n_layers_spatial=N_LAYERS_SPATIAL,
                 d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.intra_frame = IntraFrameModule(d_model)
        self.channel_emb = nn.Embedding(n_channels, d_model)
        self.time_proj = nn.Sequential(
            nn.Linear(TIME_DIM, 64), nn.GELU(), nn.Linear(64, d_model))
        self.temporal_norm = nn.LayerNorm(d_model)
        tl = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
            dim_feedforward=d_ff, dropout=dropout, batch_first=True, activation="gelu")
        self.temporal_transformer = nn.TransformerEncoder(tl, num_layers=n_layers_temporal,
            enable_nested_tensor=False)
        self.patch_pos_emb = nn.Parameter(torch.randn(1, N_PATCHES, d_model) * 0.02)
        self.spatial_norm = nn.LayerNorm(d_model)
        sl = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
            dim_feedforward=d_ff, dropout=dropout, batch_first=True, activation="gelu")
        self.spatial_transformer = nn.TransformerEncoder(sl, num_layers=n_layers_spatial,
            enable_nested_tensor=False)
        self.patch_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, PATCH_SIZE * PATCH_SIZE))

    def forward(self, channel_ids, telop_feats, telop_mask, time_feats, frame_mask):
        B, T, K, _ = telop_feats.shape
        device = telop_feats.device
        dtype = telop_feats.dtype
        BT = B * T
        sf = telop_feats.reshape(BT, K, SLOT_DIM)
        sm = telop_mask.reshape(BT, K)
        fl = []
        for s in range(0, BT, INTRA_CHUNK):
            e = min(s + INTRA_CHUNK, BT)
            fl.append(self.intra_frame(sf[s:e], sm[s:e]))
        ft = torch.cat(fl, dim=0).reshape(B, T, -1)
        ch = self.channel_emb(channel_ids).unsqueeze(1).expand(-1, T, -1)
        time = self.time_proj(time_feats)
        tokens = self.temporal_norm(ft + ch + time)
        to = self.temporal_transformer(tokens, src_key_padding_mask=~frame_mask)
        tf = to.reshape(BT, -1)
        vf = frame_mask.reshape(BT)
        vi = vf.nonzero(as_tuple=True)[0]
        nv = vi.shape[0]
        al = torch.zeros(BT, GRID_H, GRID_W, device=device, dtype=dtype)
        if nv > 0:
            vc = tf[vi]
            queries = vc.unsqueeze(1) + self.patch_pos_emb
            queries = self.spatial_norm(queries)
            cl = []
            for s in range(0, nv, SPATIAL_CHUNK):
                e = min(s + SPATIAL_CHUNK, nv)
                so = self.spatial_transformer(queries[s:e])
                cl.append(self.patch_head(so))
            ap = torch.cat(cl, dim=0)
            ml = (ap.reshape(-1, N_PATCHES_Y, N_PATCHES_X, PATCH_SIZE, PATCH_SIZE)
                  .permute(0, 1, 3, 2, 4).contiguous().reshape(-1, GRID_H, GRID_W))
            al[vi] = ml.to(al.dtype)
        return al.reshape(B, T, GRID_H, GRID_W)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model = ViTPatchMaskModel(n_channels=len(channels)).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"✅ 모델 로드: {CKPT_PATH}")


def prepare_and_infer(sample):
    channel_id = channel2id[sample["channel"]]
    instances = sample["instances"]
    duration = max(sample["duration"], 0.1)
    n_frames = max(1, min(int(duration * FPS) + 1, MAX_FRAMES))
    n_inst = len(instances)

    times = np.arange(n_frames, dtype=np.float32) / FPS
    time_feats = np.zeros((n_frames, TIME_DIM), dtype=np.float32)
    t_norm = times / max(duration, 1.0)
    time_feats[:, 0] = t_norm
    time_feats[:, 1] = np.sin(2 * np.pi * t_norm)
    time_feats[:, 2] = np.cos(2 * np.pi * t_norm)

    telop_feats = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME, SLOT_DIM), dtype=np.float32)
    telop_mask  = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME), dtype=np.bool_)

    if n_inst > 0:
        inst_starts = np.array([inst["start"] for inst in instances])
        inst_ends   = np.array([inst["end"]   for inst in instances])
        inst_tlens  = np.array([inst["text_len"] for inst in instances])

        channel_emb = text2emb.get(sample["channel"], ZERO_EMB)
        video_emb   = text2emb.get(sample["video_name"], ZERO_EMB)
        inst_embs = torch.stack([text2emb.get(inst["text"], ZERO_EMB) for inst in instances])
        ch_sims  = F.cosine_similarity(inst_embs, channel_emb.unsqueeze(0), dim=-1).numpy()
        vid_sims = F.cosine_similarity(inst_embs, video_emb.unsqueeze(0), dim=-1).numpy()

        stt_sims = np.zeros(n_inst, dtype=np.float32)
        has_stts = np.zeros(n_inst, dtype=np.float32)
        stt_segments = sample["stt_segments"]
        if len(stt_segments) > 0:
            stt_starts = np.array([seg["start"] for seg in stt_segments])
            stt_ends   = np.array([seg["end"]   for seg in stt_segments])
            stt_embs   = torch.stack([text2emb.get(seg["text"], ZERO_EMB) for seg in stt_segments])
            for i in range(n_inst):
                mid = (inst_starts[i] + inst_ends[i]) / 2
                stt_active = (stt_starts <= mid) & (stt_ends > mid)
                stt_active_idx = np.where(stt_active)[0]
                if len(stt_active_idx) > 0:
                    stt_sims[i] = F.cosine_similarity(
                        inst_embs[i].unsqueeze(0), stt_embs[stt_active_idx[0]].unsqueeze(0)).item()
                    has_stts[i] = 1.0

        active_matrix = (inst_starts[None, :] <= times[:, None] + 0.05) & (inst_ends[None, :] > times[:, None])

        for fi in range(n_frames):
            active_idx = np.where(active_matrix[fi])[0]
            if len(active_idx) > 0:
                sorted_order = np.argsort(inst_tlens[active_idx])[::-1][:MAX_ACTIVE_PER_FRAME]
                sorted_idx = active_idx[sorted_order]
                for si, ai in enumerate(sorted_idx):
                    telop_feats[fi, si, 0] = inst_tlens[ai] / MAX_TEXT_LEN
                    telop_feats[fi, si, 1] = ch_sims[ai]
                    telop_feats[fi, si, 2] = vid_sims[ai]
                    telop_feats[fi, si, 3] = stt_sims[ai]
                    telop_feats[fi, si, 4] = has_stts[ai]
                    telop_mask[fi, si] = True

    with torch.no_grad():
        cid = torch.tensor([channel_id], dtype=torch.long, device=device)
        tf_t = torch.from_numpy(telop_feats).unsqueeze(0).to(device)
        tm_t = torch.from_numpy(telop_mask).unsqueeze(0).to(device)
        ti_t = torch.from_numpy(time_feats).unsqueeze(0).to(device)
        fm_t = torch.ones(1, n_frames, dtype=torch.bool, device=device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            pred_logits = model(cid, tf_t, tm_t, ti_t, fm_t)

        pred_prob = torch.sigmoid(pred_logits[0]).cpu().half()

    return pred_prob  # (T, 80, 80) float16


# ── 채널별 추론 + 저장 ──
n_skip = 0
n_done = 0

for ch in tqdm(sorted(by_channel.keys()), desc="채널별 히트맵 생성"):
    out_path = os.path.join(HEATMAP_DIR, f"{ch}.pt")

    if os.path.exists(out_path):
        n_skip += 1
        continue

    ch_samples = by_channel[ch]
    ch_heatmaps = {}  # file_id → (T, 80, 80) float16

    for sample in ch_samples:
        try:
            heatmap = prepare_and_infer(sample)
            ch_heatmaps[sample["file_id"]] = heatmap
        except Exception as e:
            print(f"⚠️ 실패: {ch}/{sample['file_id']} → {e}")

    os.makedirs(HEATMAP_DIR, exist_ok=True)
    torch.save(ch_heatmaps, out_path)
    n_done += 1

print(f"\n✅ 완료: 생성={n_done:,}채널  스킵={n_skip:,}채널  총={len(by_channel):,}채널")

✅ 임베딩 로드: 2,167,019개


로드: 100%|██████████| 66400/66400 [00:16<00:00, 4020.26it/s]


✅ 전체 영상: 66,400개  채널: 664개
✅ 모델 로드: ./model/8_layout_vit_patch_mask_focal/epoch_012.pt


채널별 히트맵 생성: 100%|██████████| 664/664 [22:23<00:00,  2.02s/it]


✅ 완료: 생성=664채널  스킵=0채널  총=664채널


In [2]:
# %% 인스턴스별 평균 히트맵 사전 계산 → 채널별 .pt 저장 (standalone)
import os, json
import torch
import numpy as np
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
HEATMAP_DIR = "./data/8_heatmap"
INST_HMAP_DIR = "./data/8_heatmap_instance"
GRID_H = 80
GRID_W = 80
FPS = 10
NUM_WORKERS = 32

os.makedirs(INST_HMAP_DIR, exist_ok=True)

# ── 데이터 로드 (최소한만) ──
def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", [])
    file_id = os.path.basename(path)[:-5]
    inst_list = []
    for inst in instances:
        inst_list.append({
            "start": inst["start_sec"],
            "end": inst["end_sec"],
        })
    return {
        "channel": channel,
        "file_id": file_id,
        "instances": inst_list,
    }


json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

samples = []
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        if len(result["instances"]) > 0:
            samples.append(result)

print(f"✅ 텔롭 있는 영상: {len(samples):,}개")

# ── 채널별 그룹화 ──
by_channel = {}
for s in samples:
    if s["channel"] not in by_channel:
        by_channel[s["channel"]] = []
    by_channel[s["channel"]].append(s)

print(f"   채널: {len(by_channel)}개")

# ── 인스턴스별 히트맵 생성 ──
n_skip = 0
n_done = 0

for ch in tqdm(sorted(by_channel.keys()), desc="인스턴스 히트맵 생성"):
    out_path = os.path.join(INST_HMAP_DIR, f"{ch}.pt")
    if os.path.exists(out_path):
        n_skip += 1
        continue

    hmap_path = os.path.join(HEATMAP_DIR, f"{ch}.pt")
    if not os.path.exists(hmap_path):
        continue
    ch_hmaps = torch.load(hmap_path, weights_only=True)

    ch_inst_hmaps = {}

    for s in by_channel[ch]:
        fid = s["file_id"]
        raw = ch_hmaps.get(fid)
        if raw is None:
            continue

        instances = s["instances"]
        n_inst = len(instances)
        inst_hmaps = torch.zeros(n_inst, GRID_H, GRID_W, dtype=torch.float16)

        for j, inst in enumerate(instances):
            f_start = max(0, int(inst["start"] * FPS))
            f_end   = min(raw.shape[0], int(inst["end"] * FPS) + 1)
            if f_end > f_start:
                inst_hmaps[j] = raw[f_start:f_end].float().mean(dim=0).half()

        ch_inst_hmaps[fid] = inst_hmaps

    torch.save(ch_inst_hmaps, out_path)
    n_done += 1
    del ch_hmaps

print(f"\n✅ 완료: 생성={n_done:,}채널  스킵={n_skip:,}채널  총={len(by_channel):,}채널")

로드: 100%|██████████| 66400/66400 [00:15<00:00, 4203.16it/s]


✅ 텔롭 있는 영상: 59,876개
   채널: 664개


인스턴스 히트맵 생성: 100%|██████████| 664/664 [20:00<00:00,  1.81s/it]


✅ 완료: 생성=664채널  스킵=0채널  총=664채널
